# Fase 0 — Descarga del corpus (RICSMUR)

Scraper que recorre el repositorio RICSMUR del Servicio Murciano de Salud (plataforma DSpace) para descargar la colección de protocolos clínicos y guías de práctica médica.

Para cada ítem del repositorio:
1. Extrae los metadatos Dublin Core (`DC.*`, `DCTERMS.*`) directamente de las etiquetas `<meta>` del HTML.
2. Localiza el enlace al PDF dentro del `bitstream` de DSpace.
3. Descarga el PDF —con un sistema de reintentos ante fallos de red— y guarda los metadatos en un JSON junto a él.

La paginación se recorre mediante el parámetro `offset` hasta que una página deja de devolver artículos. Se incluyen pausas entre peticiones para no sobrecargar el servidor.

De la colección completa descargada aquí, los 20 documentos que forman el corpus final del trabajo se seleccionaron manualmente a posteriori para garantizar diversidad temática y representatividad clínica (ver Capítulo 4 de la memoria, apartado de Preprocesamiento del corpus).

In [1]:
import requests
from bs4 import BeautifulSoup
import os
import json
import time
from urllib.parse import urljoin

In [2]:
# CONFIGURACIÓN
BASE_URL = "https://sms.carm.es"
START_URL = "https://sms.carm.es/ricsmur/handle/123456789/187/recent-submissions"
OUTPUT_DIR = "descargas_ricsmur"

# Crear carpeta de descarga si no existe
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [3]:
def get_soup(url):
    try:
        # Petición GET con restricción de tiempo (timeout)
        response = requests.get(url, timeout=10)

        # Si el servidor da error (404 no encontrado, 500 error servidor), esto detiene el proceso
        response.raise_for_status()

        # Devuelve el objeto "sopa" listo para ser analizado
        return BeautifulSoup(response.content, 'html.parser')
    except Exception as e:
        print(f"Error accediendo a {url}: {e}")
        return None

In [4]:
def extract_metadata(soup):
    """
    Extrae metadatos directamente de las etiquetas <meta> del HTML (HEAD).
    Es más rápido y estándar que leer tablas visuales.
    """
    metadata = {}

    # Buscamos todas las etiquetas <meta>
    # Ejemplo: <meta name="DC.title" content="Protocolo..." />
    for tag in soup.find_all('meta'):
        # Obtenemos los atributos 'name' y 'content'
        name = tag.get('name')
        content = tag.get('content')

        # Filtramos: Solo nos interesan los que empiezan por DC. o DCTERMS.
        # y que tengan contenido (no vacíos)
        if name and content and (name.startswith('DC.') or name.startswith('DCTERMS.')):

            # Gestión de duplicados (ej. múltiples DC.subject)
            if name in metadata:
                if isinstance(metadata[name], list):
                    metadata[name].append(content)
                else:
                    metadata[name] = [metadata[name], content]
            else:
                metadata[name] = content

    return metadata

In [5]:
def find_pdf_link(soup, base_item_url):
    # Buscamos todas las etiquetas <a> que tengan atributo href (enlace)
    for a in soup.find_all('a', href=True):
        href = a['href']

        # RESTRICCIÓN DE FILTRADO:
        # 1. Debe contener '/bitstream/' (así organiza DSpace los archivos)
        # 2. Debe contener 'isAllowed=y' (indica que tenemos permiso público)
        if '/bitstream/' in href and 'isAllowed=y' in href:

            # RESTRICCIÓN DE SEGURIDAD:
            # A veces hay miniaturas (.jpg) o ficheros de texto.
            # Forzamos a que termine en .pdf O que sea una secuencia numerada (común en DSpace)
            if href.lower().endswith('.pdf') or 'sequence=' in href:
                 # Unimos la base (sms.carm.es) con el enlace relativo
                 return urljoin(BASE_URL, href)
    return None

In [ ]:
def main():
    # Inicializamos el offset (desplazamiento) en 0
    offset = 0

    # BUCLE DE PAGINACIÓN: Se ejecutará indefinidamente hasta que no encuentre artículos
    while True:
        # Construimos la URL dinámica: Si offset es 0 usamos la normal, si no, añadimos el parámetro
        if offset == 0:
            url_actual = START_URL
        else:
            url_actual = f"{START_URL}?offset={offset}"

        print(f"\n--- Escaneando página con offset {offset} ---")
        print(f"URL: {url_actual}")

        soup = get_soup(url_actual)

        if not soup:
            # Si falla una página, intentamos pasar a la siguiente (o podrías poner break para salir)
            print("Error cargando página, intentando siguiente bloque...")
            offset += 20
            continue

        # Buscar enlaces a los items (patrón /handle/123456789/xxxxx)
        # Evitamos enlaces de paginación o propios de la estructura
        item_links = []

        # En lugar de buscar todos los enlaces ('a'), buscamos primero los DIVs
        # que contienen los títulos de los artículos. DSpace usa la clase "artifact-title".
        artifact_divs = soup.find_all('div', class_='artifact-title')

        # CONDICIÓN DE SALIDA: Si no hay artículos en esta página, terminamos.
        if len(artifact_divs) == 0:
            print(">>> No se encontraron más artículos. Fin del proceso.")
            break

        print(f"Encontrados {len(artifact_divs)} candidatos en esta página (por etiqueta artifact-title)...")

        for div in artifact_divs:
            # Dentro de ese div específico, buscamos el enlace
            a_tag = div.find('a', href=True)

            if a_tag:
                href = a_tag['href']
                # Construimos la URL completa
                full_link = urljoin(BASE_URL, href)

                # FILTRO EXTRA DE SEGURIDAD:
                # Aunque la clase 'artifact-title' es muy específica, nos aseguramos
                # de que el enlace contenga '/handle/' para evitar sorpresas.
                if '/handle/' in full_link and full_link not in item_links:
                    item_links.append(full_link)

        print(f"Encontrados {len(item_links)} items válidos. Procesando...")

        # Bucle principal: Vamos enlace por enlace
        for link in item_links:
            print(f"\nProcesando: {link}")

            # Descargamos la página del ítem una sola vez aquí, al principio del bucle,
            # para reutilizarla tanto en la extracción de metadatos como en la búsqueda del PDF.
            item_soup = get_soup(link)

            if not item_soup:
                continue

            # Le pasamos el HTML ya descargado (item_soup) a la función, no el enlace,
            # para no tener que descargar la página por segunda vez.
            try:
                metadata = extract_metadata(item_soup)
            except Exception as e:
                print(f"Error extrayendo metadatos: {e}")
                metadata = {} # Seguimos aunque falle para intentar bajar el PDF

            # --- LIMPIEZA DE NOMBRE DE ARCHIVO ---
            handle_id = link.split('/')[-1]

            # La clave Dublin Core del título usa mayúsculas (DC.title) y puede venir
            # como lista si el ítem tiene varios valores.
            title_raw = metadata.get('DC.title', handle_id)

            # Si devuelve una lista (ej: ['Protocolo...']), cogemos el primer elemento.
            if isinstance(title_raw, list):
                title = title_raw[0]
            else:
                title = title_raw

            # Sin límite de caracteres, pero reemplazando TODOS los símbolos prohibidos en Windows
            safe_title = str(title).replace('/', '-').replace('\\', '-').replace(':', '-').replace('*', '').replace('?', '').replace('"', '').replace('<', '').replace('>', '').replace('|', '').strip()
            file_base_name = f"{handle_id}_{safe_title}"

            # Guardamos el JSON
            json_path = os.path.join(OUTPUT_DIR, f"{file_base_name}.json")
            # Opcional: Evitar reescribir si ya existe
            if not os.path.exists(json_path):
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(metadata, f, ensure_ascii=False, indent=4)
                print(f" - Metadatos guardados.")
            else:
                print(f" - Metadatos ya existentes.")

            # Reutilizamos item_soup (ya descargado) para localizar también el enlace al PDF.
            pdf_url = find_pdf_link(item_soup, link)

            pdf_path = os.path.join(OUTPUT_DIR, f"{file_base_name}.pdf")

            if pdf_url:
                if not os.path.exists(pdf_path):
                    # --- Sistema de reintentos ante fallos de descarga ---
                    max_intentos = 3
                    descarga_exitosa = False

                    for intento in range(max_intentos):
                        try:
                            if intento > 0:
                                print(f" - Reintentando descarga (Intento {intento+1})...")
                            else:
                                print(f" - Descargando PDF...")

                            # Añadimos timeout de 30 segundos por si el servidor va lento
                            pdf_resp = requests.get(pdf_url, stream=True, timeout=30)

                            with open(pdf_path, 'wb') as f:
                                for chunk in pdf_resp.iter_content(chunk_size=8192):
                                    if chunk: # Filtramos chunks vacíos de keep-alive
                                        f.write(chunk)

                            # Si llegamos aquí sin error, todo fue bien
                            print(" - PDF descargado correctamente.")
                            descarga_exitosa = True
                            break # Salimos del bucle de reintentos

                        except Exception as e:
                            print(f" - Error descarga: {e}")
                            # Si falla, borramos el archivo corrupto/a medias para que no ocupe espacio
                            if os.path.exists(pdf_path):
                                os.remove(pdf_path)

                            if intento < max_intentos - 1:
                                time.sleep(3) # Esperamos 3 segundos antes de reintentar

                    if not descarga_exitosa:
                        print(" - IMPOSIBLE descargar tras varios intentos.")
                else:
                    print(" - PDF ya existe. Saltando.")
            else:
                print(" - NO se encontró enlace PDF válido.")

            # RESTRICCIÓN DE TIEMPO
            time.sleep(1)

        # Al terminar la página actual, aumentamos el offset para la siguiente vuelta del while
        offset += 20
        # Pausa extra entre páginas para no saturar
        time.sleep(2)

In [7]:
if __name__ == "__main__":
    main()


--- Escaneando página con offset 0 ---
URL: https://sms.carm.es/ricsmur/handle/123456789/187/recent-submissions
Encontrados 20 candidatos en esta página (por etiqueta artifact-title)...
Encontrados 20 items válidos. Procesando...

Procesando: https://sms.carm.es/ricsmur/handle/123456789/19184
 - Metadatos guardados.
 - Descargando PDF...
 - PDF descargado correctamente.

Procesando: https://sms.carm.es/ricsmur/handle/123456789/18124
 - Metadatos guardados.
 - Descargando PDF...
 - PDF descargado correctamente.

Procesando: https://sms.carm.es/ricsmur/handle/123456789/16644
 - Metadatos guardados.
 - Descargando PDF...
 - PDF descargado correctamente.

Procesando: https://sms.carm.es/ricsmur/handle/123456789/16404
 - Metadatos guardados.
 - Descargando PDF...
 - PDF descargado correctamente.

Procesando: https://sms.carm.es/ricsmur/handle/123456789/16384
 - Metadatos guardados.
 - Descargando PDF...
 - PDF descargado correctamente.

Procesando: https://sms.carm.es/ricsmur/handle/123456